# Ingest constructors.json file
1. Read the file using spark dataframe reader API
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table  

In [0]:
%run ../00-common-Config/01-environment-variable

In [0]:
%run ../00-common-Config/02-helper-function

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
source_file=f"{landing_path}/{v_batch_id}/constructors.json"
table_name=f"{catalog_name}.{bronze_schema}.constructors"

In [0]:
constructors_schema="""
                    constructorId STRING,
                    name STRING,
                    nationality STRING,
                    url STRING """

In [0]:
constructors_df=spark.read.format("json")\
    .schema(constructors_schema)\
    .option('mode','FAIL_FAST')\
    .load(source_file)

constructors_df.show()


## adding the metadata column

In [0]:
constructors_df=add_ingestion_metadata(constructors_df)

In [0]:
# constructors_df.write.format("delta")\
#     .mode("overwrite")\
#     .saveAsTable(table_name)

write_to_bronze(
    input_df=constructors_df,
    target_table=table_name,
    batch_id=v_batch_id
)

In [0]:
spark.read.table(table_name).show()